In [1]:
import cv2
import numpy as np

In [2]:
def main():
     # Direct video file path
     video_path = "input.mp4"
     # Open video file
     cap = cv2.VideoCapture(video_path)
     if not cap.isOpened():
          print(f"Error: Could not open video file '{video_path}'")
          print("\nTroubleshooting tips:")
          print("1. Make sure the file exists at that location")
          print("2. Try using forward slashes instead: C:\\Users\\kamal\\OneDrive\\Desktop\\Image Processing and Computer Vision\\LAB7\\t1.mp4")
          print("3. Check if the file is corrupted")
          return
     # Read first frame to initialize background
     ret, first_frame = cap.read()
     if not ret:
          print("Error: Could not read from webcam")
          return
     # Convert to grayscale and blur for background
     prev_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
     prev_gray = cv2.GaussianBlur(prev_gray, (15, 15), 0)
     print("Motion Tracking Started...")
     print("Press 'q' to quit\n")
     while True:
          ret, frame = cap.read()
          if not ret:
               break
          # Process frame
          gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
          gray = cv2.GaussianBlur(gray, (15, 15), 0)
          # Calculate absolute difference between frames
          frame_diff = cv2.absdiff(prev_gray, gray)
          # Apply threshold to get binary image (lowered threshold for better sens
          thresh = cv2.threshold(frame_diff, 15, 255, cv2.THRESH_BINARY)[1]
          # Dilate to fill gaps and connect nearby motion areas
          thresh = cv2.dilate(thresh, None, iterations=3)
          # Find contours (motion areas)
          contours, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
          # Draw rectangles around detected motion
          motion_detected = False
          for contour in contours:
               area = cv2.contourArea(contour)
               # Lower threshold to catch more motion (cars)
               if area > 200:
                    motion_detected = True
                    x, y, w, h = cv2.boundingRect(contour)
                    # Draw thicker, more visible boxes
                    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 3)
          # Display status
          status = "MOTION DETECTED" if motion_detected else "No Motion"
          color = (0, 255, 0) if motion_detected else (0, 0, 255)
          cv2.putText(frame, status, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
          # Show frame count and contour count
          cv2.putText(frame, f"Contours: {len(contours)}", (10, 70),
          cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
          # Display the frame
          cv2.imshow("Motion Tracking", frame)
          # Optional: Show difference frame
          cv2.imshow("Frame Difference", frame_diff)
          cv2.imshow("Threshold", thresh)
          # Update previous frame
          prev_gray = gray
          # Press 'q' to quit
          if cv2.waitKey(30) & 0xFF == ord('q'):
               break
     # Release resources
     cap.release()
     cv2.destroyAllWindows()
     print("Motion Tracking Stopped")

In [3]:
if __name__ == "__main__":
     main()

Motion Tracking Started...
Press 'q' to quit

Motion Tracking Stopped
